# OFC 2026 | Residual + FiLM Transformer + SHB-aware Curvature Loss

**독립형 PyTorch 재현 노트북** — 95채널 EDFA Gain profile 예측. 기존 PI-EDFA / Transformer 소스는 변경하지 않습니다.

- Analytical Gain/Tilt/Pin baseline + Transformer residual
- Attention/FFN 경로 각각에 조건부 FiLM (Aging/Gain/Tilt)
- Masked MSE + 실제 SHB 곡률 보존형 curvature regularizer

**유의:** 공식 Feature CSV에 실측 노후도 수치가 없으면 Aging 입력은 0으로 처리하며, `Category=aging`을 학습 입력으로 사용하지 않습니다. 결과 점수는 Colab에서 실제 실행한 후에만 표시됩니다.


## 1. 실행 설정

Colab 메뉴 → 런타임 유형 변경 → **T4 GPU** 권장. 아래 `PRETRAIN_COSMOS=True`는 COSMOS 사전학습→최신 Train 미세조정, `SCRATCH_CONTROL=True`는 최신 Train만 학습한 대조 모델입니다. 전체 COSMOS 실험은 `COSMOS_MAX_ROWS=0`으로 변경합니다.

In [ ]:
PRETRAIN_COSMOS = True
COSMOS_MAX_ROWS = 4000     # 0: 전체 COSMOS 데이터, 4000: 빠른 예비 실험
PRETRAIN_EPOCHS = 25
RECENT_EPOCHS = 180
SCRATCH_CONTROL = True
CURVATURE_LAMBDA = 0.02    # 0 / 0.005 / 0.02 / 0.05 ablation
CURVATURE_TOLERANCE_DB = 0.05
LEARNING_RATE = 0.002      # 전이학습과 Scratch에 동일하게 적용
BATCH_SIZE = 64
AGE_COLUMN = None          # 실제 입력 컬럼에 노후도 측정값이 있는 경우에만 지정
POSTCONTEST_AUDIT = False # 모델 확정 전엔 False. TestGround로 모델 선택 금지


## 2. 실행 환경 & GitHub 모델 코드 가져오기

In [ ]:
import sys, subprocess, os
from pathlib import Path

def execute(cmd, cwd=None):
    print('RUN:', ' '.join(map(str,cmd)),flush=True)
    subprocess.run([str(x) for x in cmd],check=True,cwd=cwd)

import torch
print('PyTorch:',torch.__version__,'CUDA available:',torch.cuda.is_available())
execute([sys.executable,'-m','pip','-q','install','numpy','pandas','scikit-learn','matplotlib','nbformat'])
execute(['bash','-lc','git lfs version >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)'])
repo=Path('/content/LSCNS')
if not (repo/'.git').exists():
    execute(['git','clone','--depth','1','--filter=blob:none','--sparse','https://github.com/kimheeseo/LSCNS.git',str(repo)])
execute(['git','-C',str(repo),'sparse-checkout','set','26OFC_ML_challenge/FiLM_Residual_Transformer'])
source=repo/'26OFC_ML_challenge/FiLM_Residual_Transformer'
assert (source/'train.py').exists()
print('Loaded model:',source)


## 3. 공식 OFC 측정 데이터 다운로드 (Git LFS)

공식 공개 CSV를 사용합니다. TestGround 라벨은 *별도 사후 평가* 선택 시에만 다운로드합니다.

In [ ]:
data_root=Path('/content/ofc-ml-challenge-data-code')
if not (data_root/'.git').exists():
    execute(['bash','-lc',f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --filter=blob:none --sparse https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code.git {data_root}'])
execute(['git','-C',str(data_root),'sparse-checkout','set','Features'])
execute(['git','-C',str(data_root),'lfs','install','--local'])
files=[
  'Features/Train/train_features.csv','Features/Train/train_labels.csv',
  'Features/Test/test_features.csv',
]
if PRETRAIN_COSMOS:
    files += ['Features/Train/COSMOS_features.csv','Features/Train/COSMOS_labels.csv']
if POSTCONTEST_AUDIT:
    files += ['Features/TestGround/test_features.csv','Features/TestGround/test_labels.csv']
execute(['git','-C',str(data_root),'lfs','pull','--include='+','.join(files),'--exclude='])
for f in files:
    x=data_root/f
    assert x.stat().st_size>1024, 'LFS 파일 또는 데이터 오류: '+str(x)
print('Data check OK:',len(files),'CSV files')


## 4. 모델 동작 테스트 후 학습 실행

아래 테스트의 합성 데이터 결과는 대회 성적이 아닙니다. 실제 Train 결과는 별도의 `results` 폴더에 저장됩니다.

In [ ]:
execute([sys.executable,str(source/'test_pipeline.py')])
output=Path('/content/ofc_film_residual_results')
cmd=[sys.executable,str(source/'train.py'), '--data-root',str(data_root), '--output',str(output),
     '--epochs',str(RECENT_EPOCHS),'--lr',str(LEARNING_RATE), '--batch-size',str(BATCH_SIZE),
     '--curvature',str(CURVATURE_LAMBDA),'--curvature-tolerance',str(CURVATURE_TOLERANCE_DB),
     '--seed','2026']
if AGE_COLUMN:
    cmd+=['--age-column',str(AGE_COLUMN)]
if PRETRAIN_COSMOS:
    cmd+=['--pretrain-cosmos','--cosmos-max-rows',str(COSMOS_MAX_ROWS),
          '--pretrain-epochs',str(PRETRAIN_EPOCHS)]
if SCRATCH_CONTROL and PRETRAIN_COSMOS:
    cmd+=['--scratch-control']
execute(cmd)


## 5. 실제 학습 결과 비교 및 손실 그래프

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
results=json.loads((output/'comparison.json').read_text())
rows=[]
for name in ('cosmos_pretrain','recent_best','scratch_best'):
    if name in results:
        rows.append({'stage':name,**results[name]})
display(pd.DataFrame(rows).round(5))
for name in ('recent','scratch'):
    path=output/f'{name}_metrics.json'
    if path.exists():
        print('Category scores:',name)
        d=json.loads(path.read_text())
        display(pd.DataFrame([{'category':k,**v} for k,v in d.items()]).round(5))
plt.figure(figsize=(9,4.5))
for stage in ('cosmos_pretrain','recent','scratch'):
    path=output/f'{stage}_history.csv'
    if path.exists():
        history=pd.read_csv(path)
        plt.plot(history.epoch,history.val_score,label=stage)
plt.xlabel('Epoch');plt.ylabel('OFC-style Train validation score (lower is better)')
plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.show()
print('Checkpoint:',output/'recent_best.pt')
print('Submission:',output/'submission.csv')


## 6. 선택: Frozen checkpoint 사후 TestGround 평가

**모델 구조·학습률·epoch 등을 이미 확정한 후에만** `POSTCONTEST_AUDIT=True`로 설정하고 3번 데이터 준비 셀부터 다시 실행합니다. 이 셀은 저장된 체크포인트를 재학습하지 않고 TestGround만 확인합니다. 학습 후 파라미터를 이 점수로 다시 최적화하면 독립 테스트가 아니게 됩니다.

In [ ]:
if POSTCONTEST_AUDIT:
    frozen=output/'recent_best.pt'
    assert frozen.exists(),frozen
    audit_dir=output/'postcontest_audit'
    audit=[sys.executable,str(source/'train.py'), '--data-root',str(data_root),
           '--output',str(audit_dir),'--audit-checkpoint',str(frozen)]
    if AGE_COLUMN:
        audit+=['--age-column',AGE_COLUMN]
    execute(audit)
    print((audit_dir/'frozen_checkpoint_testground_audit.json').read_text()[:4000])
else:
    print('사후 TestGround 평가는 건너뛰었습니다. (독립 홀드아웃 보호)')


## 7. 학습 결과 ZIP 다운로드

학습 완료된 가중치(.pt), 최근 Train 검증 지표, per-sample 오차표, 제출용 CSV 및 비교 기록이 포함됩니다.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/ofc_film_residual_results','zip',str(output))
print('Saved:',archive)
files.download(archive)
